In [1]:
!pip install faker hdbscan sentence-transformers xgboost shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 35.9 MB/s eta 0:00:00


In [7]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# ============================================================
# CONFIG
# ============================================================

SEED = 42
np.random.seed(SEED)

OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_CUSTOMERS = 20_000
N_PAYMENTS = 100_000

START_DATE = datetime(2026, 1, 1)
END_DATE = datetime(2026, 8, 31)

ACTIONS = [
    ("Retry", "same"),
    ("WhatsApp", "same"),
    ("SMS", "same"),
    ("Email", "same"),
    ("Alternate_Method", "alternate")
]

BANKS = [
    "HDFC",
    "ICICI",
    "SBI",
    "Axis",
    "Kotak",
    "IndusInd",
    "YesBank"
]

GATEWAYS = [
    "Razorpay_G1",
    "Razorpay_G2",
    "Razorpay_G3"
]

PAYMENT_METHODS = [
    "UPI",
    "Card",
    "NetBanking",
    "Subscription"
]

DEVICE_TYPES = [
    "Android",
    "iOS",
    "Web"
]

NETWORK_TYPES = [
    "4G",
    "5G",
    "WiFi",
    "3G"
]

COMM_CHANNELS = [
    "WhatsApp",
    "SMS",
    "Email"
]

FAILURE_TYPES = [
    "INSUFFICIENT_FUNDS",
    "BANK_TIMEOUT",
    "NETWORK_ERROR",
    "EXPIRED_CARD",
    "LIMIT_EXCEEDED",
    "AUTHENTICATION_FAILED",
    "GATEWAY_ERROR",
    "USER_ABANDONED"
]


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def soft_clip(x, low, high):
    return np.clip(x, low, high)


def random_timestamp(n):
    total_seconds = int((END_DATE - START_DATE).total_seconds())

    offsets = np.random.randint(
        0,
        total_seconds,
        size=n
    )

    return [
        START_DATE + timedelta(seconds=int(x))
        for x in offsets
    ]


# ============================================================
# 1. GENERATE CUSTOMERS
# ============================================================

def generate_customers():

    print("Generating customers...")

    customer_ids = [
        f"CUST_{i:06d}"
        for i in range(1, N_CUSTOMERS + 1)
    ]

    tenure = np.random.gamma(
        shape=2.0,
        scale=180,
        size=N_CUSTOMERS
    )

    tenure = np.clip(
        tenure,
        1,
        2500
    ).astype(int)

    transaction_count = np.random.poisson(
        lam=45,
        size=N_CUSTOMERS
    ) + 2

    transaction_count = np.clip(
        transaction_count,
        2,
        200
    )

    # Heavy-tailed customer value
    historical_value = (
        np.random.lognormal(
            mean=10.2,
            sigma=1.0,
            size=N_CUSTOMERS
        )
    )

    historical_value = np.clip(
        historical_value,
        500,
        1_000_000
    )

    # Historical success rate
    success_rate = np.random.beta(
        a=18,
        b=4,
        size=N_CUSTOMERS
    )

    preferred_payment = np.random.choice(
        PAYMENT_METHODS,
        size=N_CUSTOMERS,
        p=[0.55, 0.25, 0.10, 0.10]
    )

    preferred_channel = np.random.choice(
        ["WhatsApp", "SMS", "Email"],
        size=N_CUSTOMERS,
        p=[0.50, 0.30, 0.20]
    )

    # Transparent value score
    value_score = (
        0.35 * soft_clip(tenure / 1000, 0, 1)
        + 0.30 * soft_clip(transaction_count / 100, 0, 1)
        + 0.35 * soft_clip(
            np.log1p(historical_value) / np.log1p(1_000_000),
            0,
            1
        )
    )

    value_score = (
        value_score * 100
        + np.random.normal(0, 3, N_CUSTOMERS)
    )

    value_score = soft_clip(
        value_score,
        1,
        100
    )

    tiers = pd.cut(
        value_score,
        bins=[0, 40, 70, 100],
        labels=["Low", "Medium", "High"],
        include_lowest=True
    )

    customers = pd.DataFrame({
        "customer_id": customer_ids,
        "customer_tenure_days": tenure,
        "historical_transaction_count": transaction_count,
        "historical_transaction_value": historical_value.round(2),
        "historical_success_rate": success_rate.round(3),
        "preferred_payment_method": preferred_payment,
        "preferred_communication_channel": preferred_channel,
        "customer_value_score": value_score.round(2),
        "customer_value_tier": tiers.astype(str)
    })

    return customers


# ============================================================
# 2. GENERATE PAYMENT FAILURES
# ============================================================

def generate_payments(customers):

    print("Generating payment attempts...")

    customer_sample = np.random.choice(
        customers.index,
        size=N_PAYMENTS,
        replace=True
    )

    c = customers.loc[customer_sample].reset_index(drop=True)

    transaction_ids = [
        f"TX_{i:07d}"
        for i in range(1, N_PAYMENTS + 1)
    ]

    amounts = np.random.lognormal(
        mean=6.7,
        sigma=0.9,
        size=N_PAYMENTS
    )

    amounts = np.clip(
        amounts,
        20,
        75_000
    )

    payment_method = np.random.choice(
        PAYMENT_METHODS,
        N_PAYMENTS,
        p=[0.55, 0.25, 0.10, 0.10]
    )

    banks = np.random.choice(
        BANKS,
        N_PAYMENTS
    )

    gateways = np.random.choice(
        GATEWAYS,
        N_PAYMENTS,
        p=[0.50, 0.30, 0.20]
    )

    timestamps = random_timestamp(N_PAYMENTS)

    device = np.random.choice(
        DEVICE_TYPES,
        N_PAYMENTS,
        p=[0.50, 0.25, 0.25]
    )

    network = np.random.choice(
        NETWORK_TYPES,
        N_PAYMENTS,
        p=[0.35, 0.20, 0.35, 0.10]
    )

    # --------------------------------------------------------
    # Failure generation
    # --------------------------------------------------------

    failure_reason = []

    for method in payment_method:

        if method == "Card":

            reason = np.random.choice(
                FAILURE_TYPES,
                p=[
                    0.05,  # insufficient
                    0.08,  # timeout
                    0.08,  # network
                    0.25,  # expired
                    0.15,  # limit
                    0.15,  # auth
                    0.15,  # gateway
                    0.09
                ]
            )

        elif method == "UPI":

            reason = np.random.choice(
                FAILURE_TYPES,
                p=[
                    0.20,
                    0.25,
                    0.20,
                    0.00,
                    0.15,
                    0.05,
                    0.10,
                    0.05
                ]
            )

        else:

            reason = np.random.choice(
                FAILURE_TYPES,
                p=[
                    0.15,
                    0.15,
                    0.15,
                    0.05,
                    0.15,
                    0.15,
                    0.10,
                    0.10
                ]
            )

        failure_reason.append(reason)

    failure_reason = np.array(failure_reason)

    error_map = {
        "INSUFFICIENT_FUNDS": "ERR_INSUFFICIENT_FUNDS",
        "BANK_TIMEOUT": "ERR_BANK_TIMEOUT",
        "NETWORK_ERROR": "ERR_NETWORK",
        "EXPIRED_CARD": "ERR_EXPIRED_CARD",
        "LIMIT_EXCEEDED": "ERR_LIMIT",
        "AUTHENTICATION_FAILED": "ERR_AUTH_FAILED",
        "GATEWAY_ERROR": "ERR_GATEWAY",
        "USER_ABANDONED": "ERR_USER_ABANDONED"
    }

    error_codes = [
        error_map[x]
        for x in failure_reason
    ]

    payments = pd.DataFrame({
        "transaction_id": transaction_ids,
        "customer_id": c["customer_id"].values,
        "amount": amounts.round(2),
        "payment_method": payment_method,
        "bank": banks,
        "gateway": gateways,
        "timestamp": timestamps,
        "device_type": device,
        "network_type": network,
        "error_code": error_codes,
        "failure_reason": failure_reason
    })

    return payments


# ============================================================
# 3. RECOVERY PROBABILITY SIMULATOR
# ============================================================

def base_recovery_probability(
    customer,
    payment,
    channel,
    offered_method
):

    score = -0.5

    # --------------------------------------------------------
    # Customer behaviour
    # --------------------------------------------------------

    score += (
        customer["historical_success_rate"] - 0.75
    ) * 3.0

    score += (
        customer["customer_value_score"] - 50
    ) / 100 * 0.4

    # --------------------------------------------------------
    # Failure reason
    # --------------------------------------------------------

    failure = payment["failure_reason"]

    failure_effect = {

        "INSUFFICIENT_FUNDS": -1.0,

        "BANK_TIMEOUT": 0.25,

        "NETWORK_ERROR": 0.10,

        "EXPIRED_CARD": -1.4,

        "LIMIT_EXCEEDED": -0.8,

        "AUTHENTICATION_FAILED": -0.7,

        "GATEWAY_ERROR": 0.20,

        "USER_ABANDONED": -0.5
    }

    score += failure_effect.get(
        failure,
        0
    )

    # --------------------------------------------------------
    # Payment method / alternative method
    # --------------------------------------------------------

    original_method = payment["payment_method"]

    if offered_method == "alternate":

        if original_method == "UPI":
            score += 0.65

        elif original_method == "Card":
            score += 0.70

        elif original_method == "NetBanking":
            score += 0.55

        else:
            score += 0.50

    # --------------------------------------------------------
    # Communication channel
    # --------------------------------------------------------

    if channel == "WhatsApp":

        score += 0.25

        if (
            customer["preferred_communication_channel"]
            == "WhatsApp"
        ):
            score += 0.35

    elif channel == "SMS":

        score += 0.05

        if (
            customer["preferred_communication_channel"]
            == "SMS"
        ):
            score += 0.25

    elif channel == "Email":

        score -= 0.10

        if (
            customer["preferred_communication_channel"]
            == "Email"
        ):
            score += 0.25

    elif channel == "Retry":

        score += 0.0

        # Retry is particularly bad for permanent failures
        if failure in [
            "EXPIRED_CARD",
            "AUTHENTICATION_FAILED",
            "LIMIT_EXCEEDED"
        ]:
            score -= 0.9

        # Retry is good for temporary infrastructure failures
        if failure in [
            "BANK_TIMEOUT",
            "NETWORK_ERROR",
            "GATEWAY_ERROR"
        ]:
            score += 0.6

    # --------------------------------------------------------
    # Time effects
    # --------------------------------------------------------

    hour = payment["timestamp"].hour

    if 0 <= hour <= 5:
        score -= 0.25

    if 18 <= hour <= 22:
        score += 0.10

    # --------------------------------------------------------
    # Amount effect
    # --------------------------------------------------------

    amount = payment["amount"]

    if amount > 10_000:
        score -= 0.15

    # Convert to probability
    probability = sigmoid(score)

    # Noise prevents deterministic patterns
    probability += np.random.normal(0, 0.035)

    return float(
        soft_clip(
            probability,
            0.01,
            0.98
        )
    )


# ============================================================
# 4. GENERATE COUNTERFACTUAL ACTIONS
# ============================================================

def generate_actions(customers, payments):

    print("Generating 500K counterfactual actions...")

    customer_lookup = customers.set_index(
        "customer_id"
    )

    rows = []

    action_id = 1

    for _, payment in payments.iterrows():

        customer = customer_lookup.loc[
            payment["customer_id"]
        ]

        for channel, method_type in ACTIONS:

            if channel == "Alternate_Method":

                communication = "WhatsApp"

                offered_method = "alternate"

            else:

                communication = channel

                offered_method = "same"

            # Some actions may be ineligible
            eligible = True

            if (
                payment["payment_method"] == "Subscription"
                and channel == "Alternate_Method"
            ):
                eligible = False

            # Time after failure
            time_since_failure = np.random.randint(
                60,
                1800
            )

            # Action cost
            cost_map = {
                "Retry": 0.01,
                "WhatsApp": 0.50,
                "SMS": 0.20,
                "Email": 0.05,
                "Alternate_Method": 0.25
            }

            action_cost = cost_map[channel]

            probability = base_recovery_probability(
                customer,
                payment,
                channel,
                offered_method
            )

            # Counterfactual outcome
            if eligible:

                paid = (
                    np.random.random()
                    < probability
                )

            else:

                paid = False
                probability = 0.0

            recovered_amount = (
                payment["amount"]
                if paid
                else 0.0
            )

            rows.append({
                "transaction_id":
                    payment["transaction_id"],

                "action_id":
                    f"ACT_{action_id:08d}",

                "communication_channel":
                    communication,

                "payment_method":
                    payment["payment_method"],

                "offered_payment_method":
                    offered_method,

                "attempt_number": 1,

                "action_cost":
                    action_cost,

                "time_since_failure":
                    time_since_failure,

                "eligible":
                    eligible,

                # Hidden simulator truth.
                # DO NOT expose this during training.
                "simulated_success_probability":
                    round(probability, 4),

                "actual_outcome":
                    "PAID" if paid else "NOT_PAID",

                "recovered_amount":
                    round(recovered_amount, 2)
            })

            action_id += 1

    return pd.DataFrame(rows)


# ============================================================
# 5. SELECT ACTUAL AGENT EXECUTION
# ============================================================

def generate_recovery_events(
    customers,
    payments,
    actions
):

    print("Generating recovery execution events...")

    rows = []

    attempt_id = 1

    # --------------------------------------------------------
    # Group candidate actions by transaction
    # --------------------------------------------------------

    grouped = actions.groupby(
        "transaction_id"
    )

    customer_lookup = customers.set_index(
        "customer_id"
    )

    payment_lookup = payments.set_index(
        "transaction_id"
    )

    for transaction_id, candidates in grouped:

        payment = payment_lookup.loc[
            transaction_id
        ]

        customer = customer_lookup.loc[
            payment["customer_id"]
        ]

        # ----------------------------------------------------
        # AI-style predicted probability
        # In the real pipeline this will come from the model.
        # For dataset generation we use simulated truth + noise.
        # ----------------------------------------------------

        candidate_rows = []

        for _, action in candidates.iterrows():

            probability = action[
                "simulated_success_probability"
            ]

            predicted_probability = soft_clip(
                probability + np.random.normal(0, 0.08),
                0.01,
                0.98
            )

            expected_value = (
                predicted_probability
                * payment["amount"]
                - action["action_cost"]
            )

            candidate_rows.append({
                "action": action,
                "predicted_probability":
                    predicted_probability,

                "expected_value":
                    expected_value
            })

        # ----------------------------------------------------
        # Rank candidates
        # ----------------------------------------------------

        candidate_rows.sort(
            key=lambda x: x["expected_value"],
            reverse=True
        )

        # ----------------------------------------------------
        # Triage
        # ----------------------------------------------------

        top = candidate_rows[0]

        p = top["predicted_probability"]

        ev = top["expected_value"]

        probability_floor = 0.05

        if (
            p < probability_floor
            or ev <= 0
        ):

            decision = "NOT_PURSUED"

            if p < probability_floor and ev <= 0:
                reason = "BOTH"

            elif p < probability_floor:
                reason = "PROBABILITY_FLOOR"

            else:
                reason = "NEGATIVE_EXPECTED_VALUE"

            rows.append({
                "attempt_id":
                    f"ATT_{attempt_id:08d}",

                "transaction_id":
                    transaction_id,

                "attempt_number": 0,

                "communication_channel":
                    top["action"]["communication_channel"],

                "payment_method":
                    top["action"]["payment_method"],

                "predicted_probability":
                    round(p, 4),

                "expected_value":
                    round(ev, 2),

                "decision":
                    decision,

                "decision_reason":
                    reason,

                "state":
                    "NOT_PURSUED",

                "timestamp":
                    payment["timestamp"],

                "idempotency_key":
                    f"ATT_{attempt_id:08d}_NOT_PURSUED",

                "actual_outcome":
                    "NOT_ATTEMPTED"
            })

            attempt_id += 1

            continue

        # ----------------------------------------------------
        # Sequential recovery
        # ----------------------------------------------------

        recovered = False

        max_attempts = {

            "Low": 1,
            "Medium": 2,
            "High": 3

        }.get(
            customer["customer_value_tier"],
            1
        )

        for attempt_number, candidate in enumerate(
            candidate_rows[:max_attempts],
            start=1
        ):

            action = candidate["action"]

            predicted_probability = candidate[
                "predicted_probability"
            ]

            expected_value = candidate[
                "expected_value"
            ]

            # Reconstruct actual outcome from candidate
            actual_outcome = action["actual_outcome"]

            if actual_outcome == "PAID":

                state = "RECOVERED"
                recovered = True

            else:

                if attempt_number < max_attempts:

                    state = "FAILED_RETRYABLE"

                else:

                    state = "EXPIRED_FAILED"

            rows.append({
                "attempt_id":
                    f"ATT_{attempt_id:08d}",

                "transaction_id":
                    transaction_id,

                "attempt_number":
                    attempt_number,

                "communication_channel":
                    action["communication_channel"],

                "payment_method":
                    action["payment_method"],

                "predicted_probability":
                    round(predicted_probability, 4),

                "expected_value":
                    round(expected_value, 2),

                "decision":
                    "PURSUED",

                "decision_reason":
                    "POSITIVE_EXPECTED_VALUE",

                "state":
                    state,

                "timestamp":
                    payment["timestamp"],

                "idempotency_key":
                    (
                        f"ATT_{attempt_id:08d}_"
                        f"{action['communication_channel']}_"
                        f"{attempt_number}"
                    ),

                "actual_outcome":
                    actual_outcome
            })

            attempt_id += 1

            if recovered:
                break

    return pd.DataFrame(rows)


# ============================================================
# 6. GENERATE FEEDBACK DATA
# ============================================================

def generate_feedback(events, payments):

    print("Generating feedback records...")

    payment_lookup = payments.set_index(
        "transaction_id"
    )

    rows = []

    for _, event in events.iterrows():

        transaction = payment_lookup.loc[
            event["transaction_id"]
        ]

        actual = (
            1
            if event["actual_outcome"] == "PAID"
            else 0
        )

        predicted = event[
            "predicted_probability"
        ]

        prediction_error = abs(
            actual - predicted
        )

        rows.append({

            "transaction_id":
                event["transaction_id"],

            "model_version":
                "v4.1-predictive",

            "predicted_failure":
                transaction["failure_reason"],

            "failure_probability":
                round(
                    np.random.uniform(
                        0.60,
                        0.98
                    ),
                    4
                ),

            "predicted_action":
                event["communication_channel"],

            "predicted_success_probability":
                predicted,

            "actual_action":
                event["communication_channel"],

            "actual_outcome":
                event["actual_outcome"],

            "prediction_error":
                round(
                    prediction_error,
                    4
                ),

            "timestamp":
                event["timestamp"]
        })

    return pd.DataFrame(rows)


# ============================================================
# 7. SAVE
# ============================================================

def save_datasets(
    customers,
    payments,
    actions,
    events,
    feedback
):

    print("\nSaving datasets...")

    customers.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "customers.csv"
        ),
        index=False
    )

    payments.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "payment_attempts.csv"
        ),
        index=False
    )

    actions.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "counterfactual_actions.csv"
        ),
        index=False
    )

    events.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "recovery_events.csv"
        ),
        index=False
    )

    feedback.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "feedback_loop.csv"
        ),
        index=False
    )


# ============================================================
# MAIN
# ============================================================

def main():

    start = datetime.now()

    print("=" * 70)
    print("RAZORPAY AI REVENUE RECOVERY BENCHMARK GENERATOR")
    print("=" * 70)

    customers = generate_customers()

    payments = generate_payments(
        customers
    )

    actions = generate_actions(
        customers,
        payments
    )

    events = generate_recovery_events(
        customers,
        payments,
        actions
    )

    feedback = generate_feedback(
        events,
        payments
    )

    save_datasets(
        customers,
        payments,
        actions,
        events,
        feedback
    )

    elapsed = (
        datetime.now() - start
    ).total_seconds()

    print("\n" + "=" * 70)
    print("GENERATION COMPLETE")
    print("=" * 70)

    print(
        f"Customers:              {len(customers):,}"
    )

    print(
        f"Payment attempts:       {len(payments):,}"
    )

    print(
        f"Counterfactual actions: {len(actions):,}"
    )

    print(
        f"Recovery events:        {len(events):,}"
    )

    print(
        f"Feedback records:       {len(feedback):,}"
    )

    print(
        f"\nGeneration time: {elapsed:.2f} seconds"
    )

    print(
        f"\nSaved to: {OUTPUT_DIR}/"
    )


if __name__ == "__main__":
    main()

RAZORPAY AI REVENUE RECOVERY BENCHMARK GENERATOR
Generating customers...
Generating payment attempts...
Generating 500K counterfactual actions...
Generating recovery execution events...
Generating feedback records...

Saving datasets...

GENERATION COMPLETE
Customers:              20,000
Payment attempts:       100,000
Counterfactual actions: 500,000
Recovery events:        150,989
Feedback records:       150,989

Generation time: 142.43 seconds

Saved to: data/


In [3]:
import pandas as pd
import os

DATA_DIR = "data"  # change if needed

files = {
    "customers": "customers.csv",
    "payments": "payment_attempts.csv",
    "actions": "recovery_actions.csv",
    "events": "recovery_events.csv",
    "feedback": "model_feedback.csv"
}

for name, file in files.items():

    path = os.path.join(DATA_DIR, file)

    print("\n" + "=" * 80)
    print(f"{name.upper()} DATASET")
    print("=" * 80)

    df = pd.read_csv(path)

    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nMissing values:")
    print(df.isnull().sum().loc[
        df.isnull().sum() > 0
    ].to_dict())

    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nFirst row:")
    print(df.iloc[0].to_dict())

    print("\nNumeric summary:")
    print(
        df.select_dtypes(include="number")
          .describe()
          .round(3)
          .to_string()
    )


CUSTOMERS DATASET
Shape: (20000, 9)

Columns:
['customer_id', 'customer_tenure_days', 'historical_transaction_count', 'historical_transaction_value', 'historical_success_rate', 'preferred_payment_method', 'preferred_communication_channel', 'customer_value_score', 'customer_value_tier']

Missing values:
{}

Duplicate rows: 0

First row:
{'customer_id': 'CUST_000001', 'customer_tenure_days': 430, 'historical_transaction_count': 40, 'historical_transaction_value': 52529.61, 'historical_success_rate': 0.744, 'preferred_payment_method': 'UPI', 'preferred_communication_channel': 'SMS', 'customer_value_score': 53.57, 'customer_value_tier': 'Medium'}

Numeric summary:
       customer_tenure_days  historical_transaction_count  historical_transaction_value  historical_success_rate  customer_value_score
count             20000.000                     20000.000                     20000.000                20000.000             20000.000
mean                360.205                        47.002   

In [14]:
import json
import numpy as np
import pandas as pd

# In Colab: upload the 5 CSVs via the file-browser panel and they'll land in
# /content/ by default, OR mount Drive and point this at your folder there:
#   from google.colab import drive
#   drive.mount('/content/drive')
#   DATA = "/content/drive/MyDrive/your_folder"
DATA = "/content/data" # Changed from "/content" to "/content/data"
results = []  # list of dicts: section, check, status, detail


def record(section, check, status, detail=""):
    assert status in ("PASS", "WARN", "FAIL")
    results.append({"section": section, "check": check, "status": status, "detail": detail})
    tag = {"PASS": "✅", "WARN": "⚠️ ", "FAIL": "❌"}[status]
    print(f"{tag} [{section}] {check} -- {detail}")


# ---------------------------------------------------------------------------
# Load
# ---------------------------------------------------------------------------
pa = pd.read_csv(f"{DATA}/payment_attempts.csv", parse_dates=["timestamp"])
cu = pd.read_csv(f"{DATA}/customers.csv")
ra = pd.read_csv(f"{DATA}/counterfactual_actions.csv")
re = pd.read_csv(f"{DATA}/recovery_events.csv", parse_dates=["timestamp"])
mf = pd.read_csv(f"{DATA}/feedback_loop.csv", parse_dates=["timestamp"]) # Corrected filename from model_feedback.csv to feedback_loop.csv

print("=" * 80)
print("SECTION 1 — STRUCTURAL INTEGRITY")
print("=" * 80)

# --- 1a. Uniqueness of primary keys ---------------------------------------
for df, key, name in [(pa, "transaction_id", "payment_attempts"),
                       (cu, "customer_id", "customers"),
                       (ra, "action_id", "recovery_actions"),
                       (re, "attempt_id", "recovery_events")]:
    dup = df[key].duplicated().sum()
    record("Structural", f"{name}.{key} uniqueness", "PASS" if dup == 0 else "FAIL",
           f"{dup} duplicate {key} values" if dup else f"all {len(df)} {key} values unique")

# --- 1b. Expected row counts / grain ---------------------------------------
n_txn = pa["transaction_id"].nunique()
record("Structural", "recovery_actions grain (5 rows/txn)",
       "PASS" if len(ra) == n_txn * 5 else "FAIL",
       f"{len(ra)} rows vs expected {n_txn * 5}")
record("Structural", "recovery_events grain (1 row/txn)",
       "PASS" if re["transaction_id"].is_unique and len(re) == n_txn else "FAIL",
       f"{len(re)} rows, {re['transaction_id'].nunique()} unique transaction_ids vs {n_txn} transactions")
record("Structural", "model_feedback grain (1 row/txn)",
       "PASS" if mf["transaction_id"].is_unique and len(mf) == n_txn else "FAIL",
       f"{len(mf)} rows, {mf['transaction_id'].nunique()} unique transaction_ids")

# --- 1c. Null checks on fields that must never be null ----------------------
must_not_null = {
    "payment_attempts": (pa, ["transaction_id", "customer_id", "amount", "error_code", "timestamp"]),
    "customers": (cu, ["customer_id", "customer_value_tier"]),
    "recovery_actions": (ra, ["transaction_id", "communication_channel", "true_success_probability",
                               "predicted_success_probability", "actual_outcome"]),
    "recovery_events": (re, ["transaction_id", "decision", "predicted_probability"]),
}
for name, (df, cols) in must_not_null.items():
    bad_cols = {c: int(df[c].isna().sum()) for c in cols if df[c].isna().any()}
    record("Structural", f"{name} required-field nulls", "PASS" if not bad_cols else "FAIL",
           "no nulls" if not bad_cols else f"nulls found: {bad_cols}")

# --- 1d. Range checks --------------------------------------------------------
range_checks = [
    ("recovery_actions.true_success_probability in [0,1]", ra["true_success_probability"].between(0, 1).all()),
    ("recovery_actions.predicted_success_probability in [0,1]", ra["predicted_success_probability"].between(0, 1).all()),
    ("recovery_events.predicted_probability in [0,1]", re["predicted_probability"].between(0, 1).all()),
    ("payment_attempts.amount > 0", (pa["amount"] > 0).all()),
    ("recovery_actions.action_cost >= 0", (ra["action_cost"] >= 0).all()),
    ("recovery_actions.time_since_failure > 0", (ra["time_since_failure"] > 0).all()),
]
for label, ok in range_checks:
    record("Structural", label, "PASS" if ok else "FAIL", "within bounds" if ok else "out-of-range values found")

# --- 1e. Zero-variance / degenerate columns (useless for modeling) ----------
for df, name in [(ra, "recovery_actions"), (re, "recovery_events"), (pa, "payment_attempts")]:
    for c in df.select_dtypes(include=[np.number]).columns:
        if df[c].nunique(dropna=True) <= 1:
            record("Structural", f"{name}.{c} variance", "WARN", "column is constant -- no signal")

print()
print("=" * 80)
print("SECTION 2 — REFERENTIAL INTEGRITY")
print("=" * 80)

orphan_customers = set(pa["customer_id"]) - set(cu["customer_id"])
record("Referential", "payment_attempts.customer_id -> customers", "PASS" if not orphan_customers else "FAIL",
       "all resolve" if not orphan_customers else f"{len(orphan_customers)} orphan customer_ids")

for df, name in [(ra, "recovery_actions"), (re, "recovery_events"), (mf, "model_feedback")]:
    orphan_txn = set(df["transaction_id"]) - set(pa["transaction_id"])
    record("Referential", f"{name}.transaction_id -> payment_attempts", "PASS" if not orphan_txn else "FAIL",
           "all resolve" if not orphan_txn else f"{len(orphan_txn)} orphan transaction_ids")

# every transaction should appear in every per-transaction table exactly once
missing_in_re = set(pa["transaction_id"]) - set(re["transaction_id"])
record("Referential", "every transaction has a recovery_events row", "PASS" if not missing_in_re else "FAIL",
       "complete coverage" if not missing_in_re else f"{len(missing_in_re)} transactions missing")

print()
print("=" * 80)
print("SECTION 3 — LEAKAGE AUDIT")
print("=" * 80)

# --- 3a. predicted vs true probability shouldn't be identical (that would mean
#         the "model" is just reading the ground-truth generator, not estimating it) ---
corr = ra["true_success_probability"].corr(ra["predicted_success_probability"])
mae = (ra["true_success_probability"] - ra["predicted_success_probability"]).abs().mean()
exact_match_rate = (ra["true_success_probability"] == ra["predicted_success_probability"]).mean()
status = "FAIL" if exact_match_rate > 0.01 or corr > 0.999 else "PASS"
record("Leakage", "predicted_success_probability is a noisy estimate, not a copy of ground truth", status,
       f"corr={corr:.4f}, MAE={mae:.4f}, exact-match rate={exact_match_rate:.4%}")

# --- 3b. actual_outcome should NOT be perfectly predictable from pre-decision
#         features (channel, error_code, device, network, tier, time bucket).
#         If AUC ~1.0, the simulator is a deterministic rule table wearing a
#         probability costume -- there'd be nothing left for a model to learn. ---
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

feat_df = ra.merge(pa[["transaction_id", "error_code", "device_type", "network_type", "amount"]], on="transaction_id")
feat_df = feat_df.merge(cu[["customer_id", "customer_value_tier"]],
                         left_on=feat_df["transaction_id"].map(pa.set_index("transaction_id")["customer_id"]),
                         right_on="customer_id", how="left") if "customer_id" not in feat_df.columns else feat_df

# Ensure no NaNs in the columns used for X and y before splitting
relevant_cols_for_X_and_Y = ["communication_channel", "error_code", "device_type", "network_type", "time_since_failure", "actual_outcome"]
clean_feat_df = feat_df.dropna(subset=relevant_cols_for_X_and_Y)

X = pd.get_dummies(clean_feat_df[["communication_channel", "error_code", "device_type", "network_type",
                             "time_since_failure"]], columns=["communication_channel", "error_code",
                                                               "device_type", "network_type"])
y = (clean_feat_df["actual_outcome"] == "PAID").astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=500)
clf.fit(X_tr, y_tr)
auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
if auc > 0.97:
    status, note = "FAIL", "near-perfect AUC -- outcome is deterministic from features (no noise / leakage)"
elif auc < 0.55:
    status, note = "WARN", "AUC barely above chance -- may be too little signal for a model to learn anything useful"
else:
    status, note = "PASS", "realistic signal: predictable better than chance, not deterministic"
record("Leakage", "outcome not deterministically derivable from pre-decision features", status,
       f

<>:287: SyntaxWarning: invalid escape sequence '\|'
<>:287: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_974/1037897472.py:287: SyntaxWarning: invalid escape sequence '\|'
  detail = str(r["detail"]).replace("|", "\|").replace("\n", " ")


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/model_feedback.csv'